In [33]:
import pandas as pd
import sqlite3
conn =sqlite3.connect("olist.db")
orders = pd.read_sql(
    "SELECT * FROM olist_orders_dataset",
    conn
)

items = pd.read_sql(
    "SELECT * FROM olist_order_items_dataset",
    conn
)

payments = pd.read_sql(
    "SELECT * FROM olist_order_payments_dataset",
    conn
)

reviews = pd.read_sql(
    "SELECT * FROM olist_order_reviews_dataset",
    conn
)

customers = pd.read_sql(
    "SELECT * FROM olist_customers_dataset",
    conn
)

sellers = pd.read_sql(
    "SELECT * FROM olist_sellers_dataset",
    conn
)

products = pd.read_sql(
    "SELECT * FROM olist_products_dataset",
    conn
)

geolocation = pd.read_sql(
    "SELECT * FROM olist_geolocation_dataset",
    conn
)

translation = pd.read_sql(
    "SELECT * FROM product_category_name_translation",
    conn
)

print("items:", items.shape)
print("sellers:", sellers.shape)

items: (112650, 7)
sellers: (3095, 4)


In [34]:
tables_dict = {
    "orders": orders,
    "items": items,
    "payments": payments,
    "reviews": reviews,
    "customers": customers,
    "sellers": sellers,
    "products": products,
    "geolocation": geolocation,
    "translation": translation
}

for name, df in tables_dict.items():
    print(f"{name:15} → {df.shape}")
geo_agg = (
    geolocation
    .groupby("geolocation_zip_code_prefix")
    .agg(
        representative_lat=("geolocation_lat", "median"),
        representative_lng=("geolocation_lng", "median"),
        num_geo_points=("geolocation_lat", "count")
    )
    .reset_index()
)

print("شكل geo_agg:", geo_agg.shape)
display(geo_agg.head())

orders          → (99441, 8)
items           → (112650, 7)
payments        → (103886, 5)
reviews         → (99224, 7)
customers       → (99441, 5)
sellers         → (3095, 4)
products        → (32951, 9)
geolocation     → (1000163, 5)
translation     → (71, 2)
شكل geo_agg: (19015, 4)


,geolocation_zip_code_prefix,representative_lat,representative_lng,num_geo_points
0,1001,-23.550381,-46.634027,26
1,1002,-23.548551,-46.635072,13
2,1003,-23.548977,-46.635313,17
3,1004,-23.549535,-46.634771,22
4,1005,-23.549612,-46.636532,25


In [35]:
items_sellers_geo = (
    items
    .merge(
        sellers[
            [
                "seller_id",
                "seller_zip_code_prefix",
                "seller_city",
                "seller_state"
            ]
        ],
        on="seller_id",
        how="left"
    )
    .merge(
        geo_agg,
        left_on="seller_zip_code_prefix",
        right_on="geolocation_zip_code_prefix",
        how="left"
    )
)

In [36]:
seller_geo_order = (
    items_sellers_geo
    .groupby("order_id")
    .agg(
        seller_lat=("representative_lat", "median"),
        seller_lng=("representative_lng", "median"),
        seller_geo_points=("num_geo_points", "sum")
    )
    .reset_index()
)

In [37]:
items_sellers_geo = (
    items
    .merge(
        sellers[
            [
                "seller_id",
                "seller_zip_code_prefix",
                "seller_city",
                "seller_state"
            ]
        ],
        on="seller_id",
        how="left"
    )
    .merge(
        geo_agg[
            [
                "geolocation_zip_code_prefix",
                "representative_lat",
                "representative_lng",
                "num_geo_points"
            ]
        ],
        left_on="seller_zip_code_prefix",
        right_on="geolocation_zip_code_prefix",
        how="left"
    )
)

print("شكل items_sellers_geo:", items_sellers_geo.shape)
display(
    items_sellers_geo[
        [
            "order_id",
            "seller_id",
            "seller_zip_code_prefix",
            "representative_lat",
            "representative_lng"
        ]
    ].head()
)

شكل items_sellers_geo: (112650, 14)


,order_id,seller_id,seller_zip_code_prefix,representative_lat,representative_lng
0,00010242fe8c5a6d1ba2dd792cb16214,48436dade18ac8b2bce089ec2a041202,27277,-22.498419,-44.125272
1,00018f77f2f0320c557190d7a144bdd3,dd7ddc04e1b6c2c614352b383efe2d36,3471,-23.564289,-46.519045
2,000229ec398224ef6ca0657da4fc703e,5b51032eddd242adc84c38acab88f23d,37564,-22.271648,-46.165556
3,00024acbcdf0a6daa1e931b038114c75,9d7a1d34a5052409006425275ba1c2b4,14403,-20.554951,-47.387691
4,00042b26cf59d7ce69dfabb4e55b4fd9,df560393f3a51e74553ab94004ba5c87,87900,-22.930408,-53.136438


In [38]:
seller_geo_order = (
    items_sellers_geo
    .groupby("order_id")
    .agg(
        seller_lat=("representative_lat", "median"),
        seller_lng=("representative_lng", "median"),
        seller_geo_points=("num_geo_points", "sum"),
        num_sellers=("seller_id", "nunique"),
        num_seller_states=("seller_state", "nunique"),
        num_seller_cities=("seller_city", "nunique")
    )
    .reset_index()
)

In [39]:
print("شكل seller_geo_order:", seller_geo_order.shape)

print(
    "عدد order_id المختلفة:",
    seller_geo_order["order_id"].nunique()
)

print(
    "هل order_id فريد؟",
    seller_geo_order["order_id"].is_unique
)

display(seller_geo_order.head())

شكل seller_geo_order: (98666, 7)
عدد order_id المختلفة: 98666
هل order_id فريد؟ True


,order_id,seller_lat,seller_lng,seller_geo_points,num_sellers,num_seller_states,num_seller_cities
0,00010242fe8c5a6d1ba2dd792cb16214,-22.498419,-44.125272,59.0,1,1,1
1,00018f77f2f0320c557190d7a144bdd3,-23.564289,-46.519045,39.0,1,1,1
2,000229ec398224ef6ca0657da4fc703e,-22.271648,-46.165556,71.0,1,1,1
3,00024acbcdf0a6daa1e931b038114c75,-20.554951,-47.387691,438.0,1,1,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,-22.930408,-53.136438,119.0,1,1,1


In [40]:
customer_geo = (
    customers[
        [
            "customer_id",
            "customer_zip_code_prefix"
        ]
    ]
    .merge(
        geo_agg[
            [
                "geolocation_zip_code_prefix",
                "representative_lat",
                "representative_lng",
                "num_geo_points"
            ]
        ],
        left_on="customer_zip_code_prefix",
        right_on="geolocation_zip_code_prefix",
        how="left"
    )
    .drop(columns=["geolocation_zip_code_prefix"])
    .rename(
        columns={
            "representative_lat": "customer_lat",
            "representative_lng": "customer_lng",
            "num_geo_points": "customer_geo_points"
        }
    )
)

In [41]:
print("شكل customer_geo:", customer_geo.shape)

print(
    "عدد customer_id المختلفة:",
    customer_geo["customer_id"].nunique()
)

print(
    "هل customer_id فريد؟",
    customer_geo["customer_id"].is_unique
)

display(customer_geo.head())

شكل customer_geo: (99441, 5)
عدد customer_id المختلفة: 99441
هل customer_id فريد؟ True


,customer_id,customer_zip_code_prefix,customer_lat,customer_lng,customer_geo_points
0,06b8999e2fba1a1fbc88172c00ba8bc7,14409,-20.502070,-47.396822,147.0
1,18955e83d337fd6b2def6b18a428ac77,9790,-23.727299,-46.542631,178.0
2,4e7b3e00288586ebd08712fdd0374a03,1151,-23.531294,-46.656404,103.0
3,b2b6027bc5c5109e529d4dc6358b12c3,8775,-23.497390,-46.182342,133.0
4,4f2d8ab171c80ec8364f7c12e35b23ad,13056,-22.973309,-47.141530,157.0


In [42]:
orders_with_customer_geo = orders.merge(
    customer_geo[
        [
            "customer_id",
            "customer_lat",
            "customer_lng",
            "customer_geo_points"
        ]
    ],
    on="customer_id",
    how="left"
)

print("شكل orders_with_customer_geo:",
      orders_with_customer_geo.shape)

شكل orders_with_customer_geo: (99441, 11)


In [43]:
ml_base = orders_with_customer_geo.merge(
    seller_geo_order[
        [
            "order_id",
            "seller_lat",
            "seller_lng",
            "seller_geo_points",
            "num_sellers",
            "num_seller_states",
            "num_seller_cities"
        ]
    ],
    on="order_id",
    how="left"
)

print("شكل ml_base:", ml_base.shape)

print(
    "عدد order_id:",
    ml_base["order_id"].nunique()
)

print(
    "هل order_id فريد؟",
    ml_base["order_id"].is_unique
)

شكل ml_base: (99441, 17)
عدد order_id: 99441
هل order_id فريد؟ True


In [44]:
items_agg = (
    items.groupby("order_id")
    .agg(
        total_price=("price", "sum"),
        total_freight=("freight_value", "sum"),
        num_items=("order_item_id", "count"),
        num_unique_products=("product_id", "nunique"),
        num_unique_sellers=("seller_id", "nunique")
    )
    .reset_index()
)

print("items_agg:", items_agg.shape)
display(items_agg.head())

###################################
payments_agg = (
    payments.groupby("order_id")
    .agg(
        total_payment=("payment_value", "sum"),
        num_payment_records=("payment_value", "count"),
        num_payment_types=("payment_type", "nunique")
    )
    .reset_index()
)

print("payments_agg:", payments_agg.shape)
display(payments_agg.head())

#########################################
reviews_agg = (
    reviews.groupby("order_id")
    .agg(
        avg_review_score=("review_score", "mean"),
        num_reviews=("review_id", "count")
    )
    .reset_index()
)

print("reviews_agg:", reviews_agg.shape)
display(reviews_agg.head())

#################################
items_products = items.merge(
    products,
    on="product_id",
    how="left"
)

items_products["product_volume_cm3"] = (
    items_products["product_length_cm"]
    * items_products["product_height_cm"]
    * items_products["product_width_cm"]
)

product_agg = (
    items_products.groupby("order_id")
    .agg(
        avg_product_weight_g=("product_weight_g", "mean"),
        total_product_weight_g=("product_weight_g", "sum"),
        avg_product_length_cm=("product_length_cm", "mean"),
        avg_product_height_cm=("product_height_cm", "mean"),
        avg_product_width_cm=("product_width_cm", "mean"),
        total_product_volume_cm3=("product_volume_cm3", "sum"),
        avg_product_photos_qty=("product_photos_qty", "mean"),
        avg_product_name_length=("product_name_lenght", "mean"),
        avg_product_description_length=("product_description_lenght", "mean")
    )
    .reset_index()
)

print("product_agg:", product_agg.shape)
display(product_agg.head())


items_agg: (98666, 6)


,order_id,total_price,total_freight,num_items,num_unique_products,num_unique_sellers
0,00010242fe8c5a6d1ba2dd792cb16214,58.90,13.29,1,1,1
1,00018f77f2f0320c557190d7a144bdd3,239.90,19.93,1,1,1
2,000229ec398224ef6ca0657da4fc703e,199.00,17.87,1,1,1
3,00024acbcdf0a6daa1e931b038114c75,12.99,12.79,1,1,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,199.90,18.14,1,1,1


payments_agg: (99440, 4)


,order_id,total_payment,num_payment_records,num_payment_types
0,00010242fe8c5a6d1ba2dd792cb16214,72.19,1,1
1,00018f77f2f0320c557190d7a144bdd3,259.83,1,1
2,000229ec398224ef6ca0657da4fc703e,216.87,1,1
3,00024acbcdf0a6daa1e931b038114c75,25.78,1,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04,1,1


reviews_agg: (98673, 3)


,order_id,avg_review_score,num_reviews
0,00010242fe8c5a6d1ba2dd792cb16214,5.0,1
1,00018f77f2f0320c557190d7a144bdd3,4.0,1
2,000229ec398224ef6ca0657da4fc703e,5.0,1
3,00024acbcdf0a6daa1e931b038114c75,4.0,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,5.0,1


product_agg: (98666, 10)


,order_id,avg_product_weight_g,total_product_weight_g,avg_product_length_cm,avg_product_height_cm,avg_product_width_cm,total_product_volume_cm3,avg_product_photos_qty,avg_product_name_length,avg_product_description_length
0,00010242fe8c5a6d1ba2dd792cb16214,650.0,650.0,28.0,9.0,14.0,3528.0,4.0,58.0,598.0
1,00018f77f2f0320c557190d7a144bdd3,30000.0,30000.0,50.0,30.0,40.0,60000.0,2.0,56.0,239.0
2,000229ec398224ef6ca0657da4fc703e,3050.0,3050.0,33.0,13.0,33.0,14157.0,2.0,59.0,695.0
3,00024acbcdf0a6daa1e931b038114c75,200.0,200.0,16.0,10.0,15.0,2400.0,1.0,42.0,480.0
4,00042b26cf59d7ce69dfabb4e55b4fd9,3750.0,3750.0,35.0,40.0,30.0,42000.0,1.0,59.0,409.0


In [45]:
ml_table = (
    ml_base
    .merge(items_agg, on="order_id", how="left")
    .merge(payments_agg, on="order_id", how="left")
    .merge(reviews_agg, on="order_id", how="left")
    .merge(product_agg, on="order_id", how="left")
)

print("شكل ML Table:", ml_table.shape)

شكل ML Table: (99441, 36)


In [46]:
ml_table = (
    ml_base
    .merge(items_agg, on="order_id", how="left")
    .merge(payments_agg, on="order_id", how="left")
    .merge(reviews_agg, on="order_id", how="left")
    .merge(product_agg, on="order_id", how="left")
)

print("Shape:", ml_table.shape)

print("عدد الصفوف:", len(ml_table))

print(
    "عدد order_id المختلفة:",
    ml_table["order_id"].nunique()
)

print(
    "هل order_id فريد؟",
    ml_table["order_id"].is_unique
)

print(
    "عدد الصفوف المكررة بالكامل:",
    ml_table.duplicated().sum()
)

Shape: (99441, 36)
عدد الصفوف: 99441
عدد order_id المختلفة: 99441
هل order_id فريد؟ True
عدد الصفوف المكررة بالكامل: 0


In [47]:
#ثم نتحقق من أن الدمج لم يسبب مشكلة في عدد الأعمدة
print("عدد الأعمدة:", len(ml_table.columns))
print(ml_table.columns.tolist())

#وبعدها

display(ml_table.head())
#وهناك فحص مهم جدًا هذه المرة

#بما أننا أصلحنا جغرافيا البائع، نتحقق من القيم

display(
    ml_table[
        [
            "customer_lat",
            "customer_lng",
            "seller_lat",
            "seller_lng"
        ]
    ].head(10)
)

#ثم

print(
    "عدد الصفوف التي لها نفس customer و seller latitude:",
    (
        ml_table["customer_lat"]
        == ml_table["seller_lat"]
    ).sum()
)

print(
    "عدد الصفوف التي لها نفس customer و seller longitude:",
    (
        ml_table["customer_lng"]
        == ml_table["seller_lng"]
    ).sum()
)


عدد الأعمدة: 36
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'customer_lat', 'customer_lng', 'customer_geo_points', 'seller_lat', 'seller_lng', 'seller_geo_points', 'num_sellers', 'num_seller_states', 'num_seller_cities', 'total_price', 'total_freight', 'num_items', 'num_unique_products', 'num_unique_sellers', 'total_payment', 'num_payment_records', 'num_payment_types', 'avg_review_score', 'num_reviews', 'avg_product_weight_g', 'total_product_weight_g', 'avg_product_length_cm', 'avg_product_height_cm', 'avg_product_width_cm', 'total_product_volume_cm3', 'avg_product_photos_qty', 'avg_product_name_length', 'avg_product_description_length']


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_lat,customer_lng,...,num_reviews,avg_product_weight_g,total_product_weight_g,avg_product_length_cm,avg_product_height_cm,avg_product_width_cm,total_product_volume_cm3,avg_product_photos_qty,avg_product_name_length,avg_product_description_length
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,-23.576170,-46.587276,...,1.0,500.0,500.0,19.0,8.0,13.0,1976.0,4.0,40.0,268.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,-12.126651,-45.008162,...,1.0,400.0,400.0,19.0,13.0,19.0,4693.0,1.0,29.0,178.0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,-16.744472,-48.514624,...,1.0,420.0,420.0,24.0,19.0,21.0,9576.0,1.0,46.0,232.0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,-5.774611,-35.273916,...,1.0,450.0,450.0,30.0,10.0,20.0,6000.0,3.0,59.0,468.0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,-23.675316,-46.515116,...,1.0,250.0,250.0,51.0,15.0,15.0,11475.0,4.0,38.0,316.0


,customer_lat,customer_lng,seller_lat,seller_lng
0,-23.576170,-46.587276,-23.681180,-46.444127
1,-12.126651,-45.008162,-19.807013,-43.980966
2,-16.744472,-48.514624,-21.364020,-48.228831
3,-5.774611,-35.273916,-19.836541,-43.921855
4,-23.675316,-46.515116,-23.545033,-46.261847
5,-23.551192,-50.551475,-23.468737,-46.518062
6,-27.865002,-54.474379,-23.540693,-46.711955
7,-22.805728,-43.423175,-23.114775,-46.553325
8,-27.421906,-52.674452,-21.600754,-46.893793
9,-23.474562,-47.468321,-23.484880,-46.366550


عدد الصفوف التي لها نفس customer و seller latitude: 23
عدد الصفوف التي لها نفس customer و seller longitude: 23


In [48]:
import os

os.makedirs("data/artifacts", exist_ok=True)

ml_table.to_parquet(
    "data/artifacts/01_ml_table.parquet",
    index=False
)

print("تم حفظ 01_ml_table.parquet بنجاح.")

#ثم نقرأه مرة أخيرة

check_ml = pd.read_parquet(
    "data/artifacts/01_ml_table.parquet"
)

print("Shape:", check_ml.shape)
print("هل order_id فريد؟", check_ml["order_id"].is_unique)


تم حفظ 01_ml_table.parquet بنجاح.
Shape: (99441, 36)
هل order_id فريد؟ True


In [49]:
check_ml = pd.read_parquet(
    "data/artifacts/01_ml_table.parquet"
)

print("Shape:", check_ml.shape)
print("هل order_id فريد؟", check_ml["order_id"].is_unique)

Shape: (99441, 36)
هل order_id فريد؟ True


In [50]:
print("Customer latitude missing:",
      ml_table["customer_lat"].isna().sum())

print("Customer longitude missing:",
      ml_table["customer_lng"].isna().sum())

print("Seller latitude missing:",
      ml_table["seller_lat"].isna().sum())

print("Seller longitude missing:",
      ml_table["seller_lng"].isna().sum())

Customer latitude missing: 278
Customer longitude missing: 278
Seller latitude missing: 991
Seller longitude missing: 991


In [51]:
from math import radians, sin, cos, asin, sqrt
import numpy as np

def haversine_distance(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(
        radians, [lat1, lon1, lat2, lon2]
    )

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        sin(dlat / 2) ** 2
        + cos(lat1)
        * cos(lat2)
        * sin(dlon / 2) ** 2
    )

    c = 2 * asin(sqrt(a))

    return 6371 * c

In [52]:
ml_table["seller_customer_distance_km"] = np.nan

mask = (
    ml_table["customer_lat"].notna()
    & ml_table["customer_lng"].notna()
    & ml_table["seller_lat"].notna()
    & ml_table["seller_lng"].notna()
)

ml_table.loc[mask, "seller_customer_distance_km"] = (
    ml_table.loc[mask].apply(
        lambda row: haversine_distance(
            row["customer_lat"],
            row["customer_lng"],
            row["seller_lat"],
            row["seller_lng"]
        ),
        axis=1
    )
)

In [53]:
print(
    "عدد المسافات المحسوبة:",
    ml_table["seller_customer_distance_km"].notna().sum()
)

print(
    "عدد المسافات المفقودة:",
    ml_table["seller_customer_distance_km"].isna().sum()
)

عدد المسافات المحسوبة: 98177
عدد المسافات المفقودة: 1264


In [54]:
print(
    ml_table["seller_customer_distance_km"].describe()
)

count    98177.000000
mean       601.304374
std        595.464649
min          0.000000
25%        184.974494
50%        433.374346
75%        798.118558
max       8677.859564
Name: seller_customer_distance_km, dtype: float64


In [55]:
import os

os.makedirs("data/artifacts", exist_ok=True)

ml_table.to_parquet(
    "data/artifacts/01_ml_table.parquet",
    index=False
)

print("تم حفظ ML Table النهائي.")

تم حفظ ML Table النهائي.


In [56]:
check_ml = pd.read_parquet(
    "data/artifacts/01_ml_table.parquet"
)

print("Shape:", check_ml.shape)
print("عدد order_id:", check_ml["order_id"].nunique())
print("هل order_id فريد؟", check_ml["order_id"].is_unique)
print("عدد المسافات المفقودة:",
      check_ml["seller_customer_distance_km"].isna().sum())

Shape: (99441, 37)
عدد order_id: 99441
هل order_id فريد؟ True
عدد المسافات المفقودة: 1264


In [57]:
customer_info = customers[
    [
        "customer_id",
        "customer_unique_id",
        "customer_zip_code_prefix",
        "customer_city",
        "customer_state"
    ]
]

In [58]:
ml_table = ml_table.drop(
    columns=[
        "customer_unique_id",
        "customer_zip_code_prefix",
        "customer_city",
        "customer_state"
    ],
    errors="ignore"
)

ml_table = ml_table.merge(
    customer_info,
    on="customer_id",
    how="left"
)

In [59]:
print("Shape:", ml_table.shape)
print("هل order_id فريد؟", ml_table["order_id"].is_unique)
print(ml_table.columns.tolist())

Shape: (99441, 41)
هل order_id فريد؟ True
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'customer_lat', 'customer_lng', 'customer_geo_points', 'seller_lat', 'seller_lng', 'seller_geo_points', 'num_sellers', 'num_seller_states', 'num_seller_cities', 'total_price', 'total_freight', 'num_items', 'num_unique_products', 'num_unique_sellers', 'total_payment', 'num_payment_records', 'num_payment_types', 'avg_review_score', 'num_reviews', 'avg_product_weight_g', 'total_product_weight_g', 'avg_product_length_cm', 'avg_product_height_cm', 'avg_product_width_cm', 'total_product_volume_cm3', 'avg_product_photos_qty', 'avg_product_name_length', 'avg_product_description_length', 'seller_customer_distance_km', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']


In [60]:
ml_table.to_parquet(
    "data/artifacts/01_ml_table.parquet",
    index=False
)

print("تم حفظ 01_ml_table.parquet بنجاح.")

تم حفظ 01_ml_table.parquet بنجاح.


In [61]:
check_ml = pd.read_parquet(
    "data/artifacts/01_ml_table.parquet"
)

print("Shape:", check_ml.shape)
print("هل order_id فريد؟", check_ml["order_id"].is_unique)

Shape: (99441, 41)
هل order_id فريد؟ True
